**BY**
     

*   **Shubhankar Bagchi - MT24AAC006**
*   **V Sadasiva Rao - MT24AAI168**


*   **Mohd. Deen Ul Haq - MT24AAI166**

**Under the Guidance of Dr. Saugata Sinha Sir, VNIT Nagpur**






In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

In [ ]:
# Datasetloading

df = pd.read_csv("defence_dataset.csv")

In [ ]:
# Preprocessing ( Hindi & English )

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  # links removal
    text = re.sub(r'[^a-zA-Z0-9\u0900-\u097F\s]', '', text)  # Hindi + English
    return text

df['Claims_Content'] = df['Claims_Content'].apply(clean_text)
df['Official_Content'] = df['Official_Content'].apply(clean_text)

# Combine for better learning
df['text'] = df['Claims_Content'] + " " + df['Official_Content']

In [ ]:
# Encoding

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Labels'])

In [ ]:
# Combine both columns into one

df['text'] = df['Claims_Content'].astype(str) + " " + df['Official_Content'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [ ]:
# Tokenization

max_words = 15000
max_len = 250

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
# Bi LSTM Model

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))  # fake, not related, real
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train model

history = model.fit(
    X_train_pad,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 257ms/step - accuracy: 0.5297 - loss: 1.0505 - val_accuracy: 0.6938 - val_loss: 0.9169
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 300ms/step - accuracy: 0.6562 - loss: 0.7368 - val_accuracy: 0.6938 - val_loss: 0.5372
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.6859 - loss: 0.5595 - val_accuracy: 0.7063 - val_loss: 0.5107
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 303ms/step - accuracy: 0.6703 - loss: 0.5326 - val_accuracy: 0.6750 - val_loss: 0.5236
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 253ms/step - accuracy: 0.6766 - loss: 0.5326 - val_accuracy: 0.6938 - val_loss: 0.5095
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 300ms/step - accuracy: 0.6844 - loss: 0.5288 - val_accuracy: 0.6938 - val_loss: 0.5109
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 255ms/step - accuracy: 0.6766 - loss: 0.5241 - val_accuracy: 0.6750 - val_loss: 0.5101
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 303ms/step - accuracy: 0.6906 - loss: 0.5247 - val_accuracy: 0.

In [ ]:
# Evaluation

loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

y_pred = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.6600 - loss: 0.5696
Test Accuracy: 0.6600000262260437
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step
              precision    recall  f1-score   support

           0       0.51      1.00      0.68        71
           1       1.00      0.44      0.61        68
           2       1.00      0.51      0.67        61

    accuracy                           0.66       200
   macro avg       0.84      0.65      0.65       200
weighted avg       0.83      0.66      0.65       200

[[71  0  0]
 [38 30  0]
 [30  0 31]]


In [ ]:
# Probability Calculations

def predict_with_explanation(text):

    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(pad)[0]

    labels = label_encoder.classes_

    result = {labels[i]: float(pred[i]) for i in range(len(labels))}

    predicted_label = labels[np.argmax(pred)]

    return predicted_label, result

In [ ]:
# Civilian understanding

def simplify_explanation(text, label):

    if label == "fake":
        return "⚠️ This news appears misleading. Official government sources do not support this claim. It may be exaggerated or false."

    elif label == "real":
        return "✅ This news aligns with official government information. It is likely accurate."

    else:
        return "ℹ️ This news is not directly related to Indian defence operations."

In [ ]:
def analyse_news(text):

    label, probabilities = predict_with_explanation(text)

    explanation = simplify_explanation(text, label)

    print("\n🔍 Prediction:", label)
    print("\n📊 Probabilities:")

    for k, v in probabilities.items():
        print(f"{k}: {v:.4f}")

    print("\n🧠 Explanation:")
    print(explanation)

In [ ]:
# Testing of data

news_input = input("Enter news text: ")

analyse_news(news_input)

Enter news text: india destroyed airbases of pakistan
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

🔍 Prediction: real

📊 Probabilities:
fake: 0.2305
not related: 0.2337
real: 0.5357

🧠 Explanation:
✅ This news aligns with official government information. It is likely accurate.
